Data Preparation & Train/Test Split

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/2025_parkinsons.csv")
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Missing status before recovery: {df['status'].isna().sum()}")

Loaded: 195 rows x 24 columns
Missing status before recovery: 2


Recover missing status values via subject ID

In [2]:
# Extract subject ID by stripping the trailing _<index>
df["subject_id"] = df["name"].str.rsplit("_", n=1).str[0]

# Build a subject to status lookup from rows where status is known
subject_status = (
    df.dropna(subset=["status"])
    .groupby("subject_id")["status"]
    .first()
)

# Fill missing status values
df["status"] = df["status"].fillna(df["subject_id"].map(subject_status))

print(f"Missing status after recovery: {df['status'].isna().sum()}")
print(f"Unique subjects: {df['subject_id'].nunique()}")
print(f"Class distribution: PD: {df['status'].mean():.1%}, Healthy: {(1-df['status']).mean():.1%}")

Missing status after recovery: 0
Unique subjects: 32
Class distribution: PD: 75.4%, Healthy: 24.6%


Handle remaining missing values -> split into two datasets

- Dataset A (imputed): fill NaNs with global column median — imputer fit on train only, applied to both train and test
- Dataset B (dropped): drop any row that has a NaN

In [3]:
feature_cols = [c for c in df.columns if c not in ("name", "subject_id", "status")]

# Dataset A: imputation deferred until after train/test split (fit on train only)
df_a = df.copy()

# Dataset B: deletion
df_b = df.dropna(subset=feature_cols).copy()

print(f"Dataset A (imputation deferred): {df_a.shape[0]} rows and {len(feature_cols)} features")
print(f"Dataset B (dropped):             {df_b.shape[0]} rows and {len(feature_cols)} features")

Dataset A (imputation deferred): 195 rows and 22 features
Dataset B (dropped):             187 rows and 22 features


Implement train/test split (80/20)

-> stratified on subject status to preserve the ~75 % PD / 25 % healthy ratio in both train and test

In [4]:
def subject_split(df_in, random_state=42):
    subject_labels = (
        df_in.groupby("subject_id")["status"].first().reset_index()
    )
    subjects = subject_labels["subject_id"].values
    labels   = subject_labels["status"].values

    train_subjects, test_subjects = train_test_split(
        subjects,
        test_size=0.2,
        stratify=labels,
        random_state=random_state,
    )

    train_df = df_in[df_in["subject_id"].isin(train_subjects)].copy()
    test_df  = df_in[df_in["subject_id"].isin(test_subjects)].copy()
    return train_df, test_df, train_subjects, test_subjects


def report_split(name, train_df, test_df, train_subj, test_subj):
    print(f"\n{name}")
    print(f"  Train: {len(train_subj):2d} subjects, {len(train_df):3d} recordings  "
          f"PD: {train_df['status'].mean():.1%}")
    print(f"  Test:  {len(test_subj):2d} subjects, {len(test_df):3d} recordings  "
          f"PD: {test_df['status'].mean():.1%}")


train_a, test_a, train_subj_a, test_subj_a = subject_split(df_a)
train_b, test_b, train_subj_b, test_subj_b = subject_split(df_b)

report_split("Dataset A (imputed)", train_a, test_a, train_subj_a, test_subj_a)
report_split("Dataset B (dropped)", train_b, test_b, train_subj_b, test_subj_b)

assert set(train_subj_a).isdisjoint(set(test_subj_a)), "Subject leakage in Dataset A"
assert set(train_subj_b).isdisjoint(set(test_subj_b)), "Subject leakage in Dataset B"

print("\nSubject leakage check passed.")
print(f"  Dataset A, train: {len(set(train_subj_a))} unique subjects, "
      f"test: {len(set(test_subj_a))} unique subjects")
print(f"  Dataset B, train: {len(set(train_subj_b))} unique subjects, "
      f"test: {len(set(test_subj_b))} unique subjects")


Dataset A (imputed)
  Train: 25 subjects, 152 recordings  PD: 76.3%
  Test:   7 subjects,  43 recordings  PD: 72.1%

Dataset B (dropped)
  Train: 25 subjects, 146 recordings  PD: 77.4%
  Test:   7 subjects,  41 recordings  PD: 70.7%

Subject leakage check passed.
  Dataset A, train: 25 unique subjects, test: 7 unique subjects
  Dataset B, train: 25 unique subjects, test: 7 unique subjects


In [5]:
from sklearn.impute import SimpleImputer

# Fit on train_a only, then transform both splits — no label leakage
imputer_a = SimpleImputer(strategy="median")
imputer_a.fit(train_a[feature_cols])

train_a[feature_cols] = imputer_a.transform(train_a[feature_cols])
test_a[feature_cols]  = imputer_a.transform(test_a[feature_cols])

print(f"Imputed Dataset A, train NaNs: {train_a[feature_cols].isna().sum().sum()}, "
      f"test NaNs: {test_a[feature_cols].isna().sum().sum()}")

Imputed Dataset A, train NaNs: 0, test NaNs: 0


In [6]:
save_cols = ["name"] + feature_cols + ["status"]

train_a[save_cols].to_csv("../data/train_a.csv", index=False)
test_a[save_cols].to_csv("../data/test_a.csv",   index=False)
train_b[save_cols].to_csv("../data/train_b.csv", index=False)
test_b[save_cols].to_csv("../data/test_b.csv",   index=False)

print("Saved:")
print(f"  data/train_a.csv: {len(train_a)} rows (imputed)")
print(f"  data/test_a.csv: {len(test_a)} rows (imputed)")
print(f"  data/train_b.csv: {len(train_b)} rows (dropped)")
print(f"  data/test_b.csv: {len(test_b)} rows (dropped)")

Saved:
  data/train_a.csv: 152 rows (imputed)
  data/test_a.csv: 43 rows (imputed)
  data/train_b.csv: 146 rows (dropped)
  data/test_b.csv: 41 rows (dropped)
